# Projeto Final de PLN – Baseline + SVM + PTT5

**Tema:** classificação por **macroárea de gasto** + sumarização cidadã.

**Classificação (padrão do repo):** compara **TF-IDF + LogReg** (baseline) e **TF-IDF + SVM** no mesmo corpus (`objeto_html`, split 70/15/15). Treino oficial: `make train-baseline` / `make train-svm`.

**Sumarização (Fase 3):** PTT5 abstrativo + baseline extrativo.


# 1. Contexto e problema real

A Administração Pública publica diariamente grande volume de editais de licitação. Esses documentos são essenciais para garantir publicidade, concorrência e controle social. Entretanto, a linguagem dos editais tende a ser extensa, técnica e jurídica, criando uma barreira de compreensão para cidadãos, pequenos fornecedores e microempresas.

O problema real investigado neste projeto é: **como usar PLN para transformar editais de contratação pública em resumos executivos claros, acessíveis e úteis para a sociedade?**

A hipótese do grupo é que técnicas modernas de PLN, especialmente modelos Transformer e LLMs, conseguem reduzir a complexidade textual dos editais sem eliminar informações essenciais. O produto esperado não substitui o edital oficial, mas funciona como uma camada de acesso inicial, facilitando a compreensão do objeto da contratação, do órgão responsável, dos potenciais interessados e das condições principais.

## Pergunta de pesquisa

> Modelos de PLN conseguem gerar resumos de editais de contratação pública em linguagem cidadã com fidelidade suficiente ao documento original e clareza superior ao texto bruto?

## Potencial impacto aplicado

- Aumentar a transparência ativa nas contratações públicas.
- Facilitar o controle social por cidadãos não especialistas.
- Apoiar micro e pequenas empresas na triagem de oportunidades.
- Criar uma ferramenta de comunicação pública alinhada ao princípio da linguagem simples.


# 2. Alinhamento com os requisitos do projeto final

Este projeto se enquadra na **Modalidade 2 — PLN no Setor Público**, pois utiliza dados textuais de licitações públicas e aplica técnicas de Processamento de Linguagem Natural para resolver um problema real de transparência administrativa.

O trabalho atende aos principais requisitos esperados:

- identificação de um problema real tratável por PLN;
- uso de dados textuais coletados a partir de fonte pública;
- aplicação de técnicas como scraping, extração de texto, sumarização, LLMs, Transformers e engenharia de prompt;
- avaliação dos resultados com métricas automáticas e avaliação qualitativa/manual;
- discussão de limitações, riscos e uso responsável da IA;
- organização do repositório com notebook, scripts, dados e documentação.

## Entregáveis sugeridos

1. Notebook `.ipynb` com o pipeline completo.
2. Script de coleta dos editais.
3. Arquivo com a base tratada.
4. Arquivo com os resumos gerados.
5. README do repositório.
6. Slides em PDF para apresentação final.


# 3. Dados

Corpus oficial: `data/processed/licitacoes_corpus.jsonl` (423 editais). O CSV `licitacoes2025.csv` complementa exploração, coleta e sumarização.

**Modelos de classificação (mesmo split, mesmo `objeto_html`):**
- Baseline: TF-IDF + LogReg (`docs/FASE1-CLASSIFICACAO.md`)
- Comparativo: TF-IDF + SVM linear (`python scripts/run_train.py --model svm`)
- Fase 2: BERTimbau (`docs/FASE2-CLASSIFICACAO.md`)

Sumarização: PTT5 neste notebook; extrativo em `scripts/run_train.py --task summarization`.


In [ ]:
# ============================================================
# 1. Importações e configuração geral
# ============================================================

from pathlib import Path
import re
import json
import sys
import time
import unicodedata
from urllib.parse import urlparse

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# Raiz do repositório (notebooks/ → raiz)
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SEED = 42
np.random.seed(SEED)

# --- Classificação: mesmo padrão de configs/classification.yaml ---
TEXT_FIELD = "objeto_html"
VAL_SIZE = 0.15
TEST_SIZE = 0.15
CORPUS_PATH = ROOT / "data" / "processed" / "licitacoes_corpus.jsonl"

# Repositório público do grupo (sumarização / CSV exploratório)
REPO_URL = "https://github.com/primodeckers/deep-learning-pln-project"
CSV_URL = (
    "https://raw.githubusercontent.com/"
    "primodeckers/deep-learning-pln-project/main/data/raw/licitacoes2025.csv"
)

# Modelo de sumarização (Fase 3)
PTT5_MODEL_NAME = "recogna-nlp/ptt5-base-summ"

RUN_BASELINE_TRAINING = True
RUN_SVM_TRAINING = True
RUN_PTT5_SUMMARIZATION = True

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw_editais"
PROCESSED_DIR = DATA_DIR / "processed"

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)
print("Corpus oficial:", CORPUS_PATH)
print("Classificação: TF-IDF + LogReg (baseline) e TF-IDF + SVM (comparativo)")
print("Campo de texto:", TEXT_FIELD)
print("Split: 70/15/15 | seed:", SEED)
print("Sumarização com:", PTT5_MODEL_NAME)



# 4. Carregamento e inspeção da planilha de licitações

A planilha possui uma primeira linha de título e, a partir da segunda linha, o cabeçalho real das colunas. Por isso, a leitura usa `skiprows=1` e separador `;`.

Nesta etapa, o objetivo é carregar a base, remover colunas vazias, padronizar os nomes das colunas e verificar o volume inicial de registros.


In [ ]:
# ============================================================
# 2. Leitura robusta da base diretamente do GitHub
# ============================================================

def normalizar_nome_coluna(col):
    col = str(col).strip()
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
    col = col.lower()
    col = re.sub(r'[^a-z0-9]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col


def carregar_licitacoes(url=CSV_URL):
    """Carrega a base `licitacoes2025.csv` diretamente do GitHub.

    Observação técnica:
    - o arquivo usa separador `;`;
    - a primeira linha é um título da base;
    - a segunda linha contém o cabeçalho real.
    """
    df = pd.read_csv(
        url,
        sep=';',
        skiprows=1,
        encoding='utf-8-sig',
        dtype=str,
        engine='python'
    )

    df = df.loc[:, ~df.columns.astype(str).str.contains('^Unnamed', regex=True)]
    df.columns = [normalizar_nome_coluna(c) for c in df.columns]
    df = df.dropna(how='all').copy()

    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

    return df


licitacoes = carregar_licitacoes(CSV_URL)

print('Formato da base:', licitacoes.shape)
print('Colunas:', list(licitacoes.columns))

display(licitacoes.head())


### 5. Limpeza inicial e criação de variáveis tabulares

Antes da extração do texto integral dos editais, a própria planilha já permite construir variáveis úteis para análise e modelagem. Nesta etapa, foi aplicada uma limpeza inicial no campo do objeto da licitação, com remoção de expressões repetitivas, normalização de espaços e, principalmente, exclusão explícita do nome do órgão e de prefixos como “Órgão:”, a fim de reduzir vazamento de informação na tarefa de classificação por macroárea. O resultado dessa etapa é a variável `objeto_limpo`, que preserva o conteúdo substantivo do edital sem carregar diretamente a identidade institucional do órgão.

Também foram criadas variáveis tabulares auxiliares. A variável `total_homologado_num` converte o valor homologado para formato numérico, permitindo futuras análises quantitativas. A variável `tem_link_edital` indica se o registro possui link válido para o edital completo, o que é importante para a etapa de coleta textual. Já a variável `dominio_edital` identifica o domínio da URL, permitindo mapear as principais fontes de coleta e avaliar a heterogeneidade dos portais utilizados.

Essas variáveis cumprem duas funções no projeto. Primeiro, ajudam a descrever e auditar a base de dados. Segundo, podem ser usadas como contexto complementar em etapas posteriores, especialmente na sumarização, sem comprometer a integridade metodológica da classificação textual, já que o nome do órgão foi removido do conteúdo textual usado como entrada do classificador.

In [ ]:
# ============================================================
# 3. Limpeza inicial — versão anti-vazamento
# ============================================================

def normalizar_texto(txt):
    if pd.isna(txt):
        return ''
    txt = str(txt)
    txt = unicodedata.normalize('NFKD', txt).encode('ascii', 'ignore').decode('utf-8')
    txt = re.sub(r'\s+', ' ', txt)
    return txt.strip()


def gerar_variantes_orgao(orgao):
    """
    Gera variantes do nome do órgão para remoção robusta do texto.
    """
    orgao_original = '' if pd.isna(orgao) else str(orgao).strip()
    orgao_norm = normalizar_texto(orgao_original)

    variantes = set()

    for item in [orgao_original, orgao_norm]:
        if item:
            variantes.add(item)
            variantes.add(item.upper())
            variantes.add(item.lower())

    # Remove prefixos administrativos muito comuns para facilitar match
    bases = list(variantes)
    for item in bases:
        limpo = re.sub(
            r'^(EDF\s*-\s*|GDF\s*-\s*|SECRETARIA DE ESTADO DE\s+|SECRETARIA\s+DE\s+|ADMINISTRACAO REGIONAL DO\s+|ADMINISTRACAO REGIONAL DE\s+|ADMINISTRAÇÃO REGIONAL DO\s+|ADMINISTRAÇÃO REGIONAL DE\s+)',
            '',
            item,
            flags=re.IGNORECASE
        ).strip()
        if limpo:
            variantes.add(limpo)

    variantes = [v.strip() for v in variantes if v and v.strip()]
    variantes = sorted(set(variantes), key=len, reverse=True)

    return variantes


def remover_orgao_do_texto(texto, orgao):
    """
    Remove explicitamente o nome do órgão e padrões do tipo 'Órgão: ...'
    """
    if pd.isna(texto):
        return ''

    texto = str(texto)

    # Remove linhas / prefixos explícitos de órgão
    texto = re.sub(r'Órgão\s*:\s*[^\n\r.]+[.\n\r]?', ' ', texto, flags=re.IGNORECASE)
    texto = re.sub(r'Orgao\s*:\s*[^\n\r.]+[.\n\r]?', ' ', texto, flags=re.IGNORECASE)

    # Remove variantes do órgão
    for variante in gerar_variantes_orgao(orgao):
        padrao = re.escape(variante)
        texto = re.sub(padrao, ' ', texto, flags=re.IGNORECASE)

    # Normalizações finais
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'\s+([,.;:])', r'\1', texto)
    texto = re.sub(r'^[,.;:\-\s]+', '', texto)
    texto = re.sub(r'[,.;:\-\s]+$', '', texto)

    return texto.strip()


def limpar_texto_basico(texto, orgao=None):
    if pd.isna(texto):
        return ''

    texto = str(texto)

    # Remove prefixos comuns
    texto = re.sub(r'^\s*OBJETO\s*:\s*', '', texto, flags=re.IGNORECASE)
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Remove órgão explicitamente
    texto = remover_orgao_do_texto(texto, orgao)

    return texto.strip()


def valor_brasileiro_para_float(valor):
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    valor = re.sub(r'[^0-9,.-]', '', valor)
    if not valor:
        return np.nan
    valor = valor.replace('.', '').replace(',', '.')
    try:
        return float(valor)
    except ValueError:
        return np.nan


def extrair_dominio(url):
    if pd.isna(url):
        return np.nan
    try:
        return urlparse(str(url)).netloc
    except Exception:
        return np.nan


# Limpeza do objeto SEM o nome do órgão
licitacoes['objeto_limpo'] = licitacoes.apply(
    lambda row: limpar_texto_basico(row.get('objeto', ''), row.get('orgao', '')),
    axis=1
)

licitacoes['total_homologado_num'] = licitacoes['total_homologado'].apply(valor_brasileiro_para_float)
licitacoes['tem_link_edital'] = licitacoes['edital'].notna() & licitacoes['edital'].astype(str).str.startswith('http', na=False)
licitacoes['dominio_edital'] = licitacoes['edital'].apply(extrair_dominio)

print('Registros com link de edital:', int(licitacoes['tem_link_edital'].sum()))
print('Domínios mais frequentes:')
display(licitacoes['dominio_edital'].value_counts().head(10))

display(
    licitacoes[
        ['no_da_licitacao', 'modalidade', 'situacao', 'orgao', 'tipo', 'objeto_limpo', 'edital', 'total_homologado_num']
    ].head()
)

# 6. Coleta dos editais por scraping ou download

A coleta deve ser feita de forma responsável: usar `timeout`, limitar a amostra inicial durante testes, salvar os documentos brutos para reprodutibilidade, registrar falhas de download e respeitar boas práticas de acesso ao portal consultado.

A função abaixo tenta baixar o conteúdo do link do edital e salvar em disco. Em alguns portais, o link pode levar a uma página intermediária, e não diretamente ao PDF. Por isso, o código trata tanto conteúdo HTML quanto PDF.


In [ ]:
# ============================================================
# 4. Download controlado dos editais
# ============================================================

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; projeto-academico-pln-editais/1.0)'
}


def extensao_por_content_type(content_type, url):
    content_type = (content_type or '').lower()
    if 'pdf' in content_type or str(url).lower().endswith('.pdf'):
        return '.pdf'
    if 'html' in content_type or 'text' in content_type:
        return '.html'
    return '.bin'


def baixar_edital(url, licitacao_id, pasta=RAW_DIR, sleep=0.5):
    # Baixa um edital e retorna metadados do download.
    meta = {
        'licitacao_id': licitacao_id,
        'url': url,
        'status_code': None,
        'content_type': None,
        'arquivo_local': None,
        'erro': None
    }

    try:
        time.sleep(sleep)
        resp = requests.get(url, headers=HEADERS, timeout=30, allow_redirects=True)
        meta['status_code'] = resp.status_code
        meta['content_type'] = resp.headers.get('Content-Type', '')

        if resp.status_code != 200:
            meta['erro'] = f'HTTP {resp.status_code}'
            return meta

        ext = extensao_por_content_type(meta['content_type'], url)
        nome_seguro = re.sub(r'[^a-zA-Z0-9_-]+', '_', str(licitacao_id))[:80]
        path = pasta / f'{nome_seguro}{ext}'
        path.write_bytes(resp.content)

        meta['arquivo_local'] = str(path)
        return meta

    except Exception as e:
        meta['erro'] = repr(e)
        return meta


amostra_links = licitacoes[licitacoes['tem_link_edital']].head(3).copy()

# Descomente para testar o download em pequena escala.
# resultados_download = []
# for _, row in tqdm(amostra_links.iterrows(), total=len(amostra_links)):
#     resultados_download.append(baixar_edital(row['edital'], row['n_da_licitacao']))
# downloads_df = pd.DataFrame(resultados_download)
# display(downloads_df)


# 7. Extração de texto dos documentos

Depois do download, o próximo passo é extrair o texto dos editais. O pipeline considera dois cenários:

1. **PDF:** extração com `pypdf`.
2. **HTML:** extração com `BeautifulSoup`.

Em um projeto real, essa etapa provavelmente exigirá ajustes conforme o portal, pois alguns editais podem estar em páginas intermediárias, arquivos compactados, anexos ou PDFs escaneados. Para PDFs escaneados, seria necessário OCR, o que pode ser tratado como limitação ou etapa futura.


In [ ]:
# ============================================================
# 5. Extração de texto de PDF e HTML
# ============================================================

def extrair_texto_pdf(path):
    try:
        reader = PdfReader(str(path))
        textos = []
        for page in reader.pages:
            textos.append(page.extract_text() or '')
        return '\n'.join(textos).strip()
    except Exception:
        return ''


def extrair_texto_html(path):
    try:
        html = Path(path).read_text(encoding='utf-8', errors='ignore')
        soup = BeautifulSoup(html, 'html.parser')

        for tag in soup(['script', 'style', 'noscript']):
            tag.extract()

        texto = soup.get_text(separator=' ')
        texto = re.sub(r'\s+', ' ', texto).strip()
        return texto
    except Exception:
        return ''


def extrair_texto_arquivo(path):
    path = Path(path)
    if not path.exists():
        return ''
    if path.suffix.lower() == '.pdf':
        return extrair_texto_pdf(path)
    if path.suffix.lower() in ['.html', '.htm']:
        return extrair_texto_html(path)
    return ''


# Exemplo de uso após downloads:
# downloads_df['texto_extraido'] = downloads_df['arquivo_local'].apply(extrair_texto_arquivo)
# display(downloads_df[['licitacao_id', 'arquivo_local', 'texto_extraido']].head())


# 8. Pré-processamento textual

O texto do edital costuma conter elementos repetitivos: cabeçalhos, rodapés, numeração de páginas, excesso de espaços e referências administrativas. O pré-processamento busca reduzir ruído sem eliminar conteúdo juridicamente relevante.

Nesta versão do notebook, o pré-processamento também tem uma função metodológica central: **evitar vazamento de informação** na classificação por macroárea. Por isso, o nome do órgão é removido não apenas do campo `objeto`, mas também do texto integral extraído do edital, sempre que disponível. O texto usado pelo classificador é então reconstruído sem a coluna `orgao`, usando apenas conteúdo do objeto, modalidade, situação e trecho do edital já limpo.

Além disso, o notebook executa uma auditoria explícita para verificar se o nome do órgão ainda aparece no texto de classificação. Se houver vazamento residual, os registros são sinalizados e podem ser removidos antes do treino do classificador.

In [ ]:
# ============================================================
# 6. Pré-processamento textual — versão anti-vazamento
# ============================================================

def limpar_texto_edital(texto, orgao=None):
    if pd.isna(texto):
        return ''

    texto = str(texto)
    texto = texto.replace('\x00', ' ')
    texto = re.sub(r'Página\s+\d+\s+de\s+\d+', ' ', texto, flags=re.I)
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Remove nome do órgão do texto integral também
    texto = remover_orgao_do_texto(texto, orgao)

    return texto.strip()


def limitar_texto(texto, max_chars=12000):
    texto = '' if pd.isna(texto) else str(texto)
    texto = re.sub(r'\s+', ' ', texto).strip()

    if len(texto) <= max_chars:
        return texto

    metade = max_chars // 2
    return texto[:metade] + '\n[...]\n' + texto[-metade:]


possiveis_colunas_texto_edital = [
    'texto_edital',
    'texto_extraido',
    'texto_documento',
    'conteudo_edital',
    'corpus_texto'
]

coluna_texto_edital = None
for col in possiveis_colunas_texto_edital:
    if col in licitacoes.columns:
        coluna_texto_edital = col
        break

if coluna_texto_edital is not None:
    licitacoes['texto_edital_limpo'] = licitacoes.apply(
        lambda row: limpar_texto_edital(row[coluna_texto_edital], row.get('orgao', '')),
        axis=1
    )
else:
    licitacoes['texto_edital_limpo'] = ''

# Texto para sumarização: aqui PODE ter órgão, porque resumir edital
# não é a tarefa alvo da classificação.
licitacoes['texto_minimo_para_resumo'] = (
    'Órgão: ' + licitacoes['orgao'].fillna('') + '\n' +
    'Modalidade: ' + licitacoes['modalidade'].fillna('') + '\n' +
    'Situação: ' + licitacoes['situacao'].fillna('') + '\n' +
    'Tipo: ' + licitacoes['tipo'].fillna('') + '\n' +
    'Objeto: ' + licitacoes['objeto_limpo'].fillna('') + '\n' +
    'Valor homologado: ' + licitacoes['total_homologado'].fillna('não informado')
)

# Texto para classificação: SEM órgão, sem prefixos do órgão, sem coluna orgao.
licitacoes['texto_para_classificacao_area'] = (
    'Modalidade: ' + licitacoes['modalidade'].fillna('não informada') + '. ' +
    'Situação: ' + licitacoes['situacao'].fillna('não informada') + '. ' +
    'Objeto: ' + licitacoes['objeto_limpo'].fillna('') + '. ' +
    'Texto do edital: ' + licitacoes['texto_edital_limpo'].fillna('').apply(lambda x: limitar_texto(x, max_chars=4000))
).str.strip()

licitacoes['texto_para_classificacao_area'] = (
    licitacoes['texto_para_classificacao_area']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print('Exemplo de texto para resumo:')
print(licitacoes['texto_minimo_para_resumo'].iloc[0][:800])

print('\nExemplo de texto para classificação por macroárea (sem órgão):')
print(licitacoes['texto_para_classificacao_area'].iloc[0][:800])

### 8.1 Auditoria anti-vazamento

A célula abaixo verifica explicitamente se o nome do órgão ainda aparece no texto usado pelo classificador. O objetivo é detectar vazamento residual antes do treino. Se houver ocorrência, os registros podem ser exibidos para inspeção e removidos da etapa supervisionada.

In [ ]:
# ============================================================
# 6.1 Auditoria anti-vazamento
# ============================================================

def texto_contem_orgao(texto, orgao):
    if pd.isna(texto) or pd.isna(orgao):
        return False

    texto_norm = normalizar_texto(texto).lower()
    variantes = [normalizar_texto(v).lower() for v in gerar_variantes_orgao(orgao)]

    for v in variantes:
        if v and v in texto_norm:
            return True

    return False

licitacoes['vazamento_orgao_objeto'] = licitacoes.apply(
    lambda row: texto_contem_orgao(row['objeto_limpo'], row.get('orgao', '')),
    axis=1
)

licitacoes['vazamento_orgao_classificacao'] = licitacoes.apply(
    lambda row: texto_contem_orgao(row['texto_para_classificacao_area'], row.get('orgao', '')),
    axis=1
)

print('Registros com vazamento do órgão em objeto_limpo:', int(licitacoes['vazamento_orgao_objeto'].sum()))
print('Registros com vazamento do órgão em texto_para_classificacao_area:', int(licitacoes['vazamento_orgao_classificacao'].sum()))

if licitacoes['vazamento_orgao_classificacao'].sum() > 0:
    print('\nExemplos com possível vazamento residual:')
    display(
        licitacoes.loc[
            licitacoes['vazamento_orgao_classificacao'],
            ['no_da_licitacao', 'orgao', 'texto_para_classificacao_area']
        ].head(10)
    )
else:
    print('\nAuditoria concluída: não foi detectado nome do órgão no texto de classificação.')

# 9. Classificação — baseline TF-IDF + LogReg

Mesmo pipeline de `scripts/run_train.py --model baseline` e `configs/classification.yaml`.

Referência: F1 macro teste ≈ **0,74** (`classification_baseline_20260608-190839`).


In [ ]:
# ============================================================
# 7. Classificação por macroárea — baseline TF-IDF + LogReg
# (mesmo código que scripts/run_train.py --model baseline)
# ============================================================

import matplotlib.pyplot as plt

from src.evaluate.metrics_classification import compute_metrics, format_metrics
from src.models.baseline_tfidf import build_baseline
from src.preprocess.dataset import make_dataset
from src.preprocess.labels import AREAS

if not CORPUS_PATH.is_file():
    raise FileNotFoundError(
        f"Corpus não encontrado: {CORPUS_PATH}\n"
        "Clone o repositório completo ou rode run_preprocess.py."
    )

dataset = make_dataset(
    CORPUS_PATH,
    text_field=TEXT_FIELD,
    seed=SEED,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
)

baseline_params = {
    "ngram_max": 2,
    "min_df": 2,
    "max_features": 20000,
    "C": 1.0,
    "max_iter": 1000,
    "class_weight": "balanced",
    "seed": SEED,
}

pipeline_baseline = build_baseline(**baseline_params)

if RUN_BASELINE_TRAINING:
    print("\nTreinando baseline TF-IDF + LogReg...")
    pipeline_baseline.fit(dataset.train.texts, dataset.train.labels)
else:
    print("RUN_BASELINE_TRAINING=False: pulando treinamento.")


def avaliar_split(nome: str, split) -> dict:
    y_pred = pipeline_baseline.predict(split.texts)
    metrics = compute_metrics(split.labels, list(y_pred), AREAS)
    return {
        "conjunto": nome,
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
    }


desempenho = pd.DataFrame(
    [
        avaliar_split("treino", dataset.train),
        avaliar_split("validação", dataset.val),
        avaliar_split("teste", dataset.test),
    ]
)

print(f"\nTamanho treino: {len(dataset.train)}")
print(f"Tamanho validação: {len(dataset.val)}")
print(f"Tamanho teste: {len(dataset.test)}")
display(desempenho)

y_test_pred = pipeline_baseline.predict(dataset.test.texts)
test_metrics = compute_metrics(
    dataset.test.labels, list(y_test_pred), AREAS
)
print("\n" + format_metrics(test_metrics))

# Matriz de confusão (teste)
cm = np.array(test_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Matriz de confusão — baseline (teste)")
ax.set_xticks(range(len(AREAS)))
ax.set_yticks(range(len(AREAS)))
ax.set_xticklabels(AREAS, rotation=45, ha="right")
ax.set_yticklabels(AREAS)
ax.set_xlabel("Predito")
ax.set_ylabel("Verdadeiro")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 7.1 Termos mais relevantes por macroárea (TF-IDF + LogReg)
# ============================================================

if RUN_BASELINE_TRAINING and "pipeline_baseline" in globals():
    tfidf_vec = pipeline_baseline.named_steps["tfidf"]
    logreg = pipeline_baseline.named_steps["clf"]
    feature_names = np.array(tfidf_vec.get_feature_names_out())

    print("Termos mais relevantes por macroárea (coeficientes LogReg):\n")

    for i, area in enumerate(AREAS):
        top_idx = np.argsort(logreg.coef_[i])[::-1][:15]
        termos = ", ".join(feature_names[top_idx])
        print(f"{area}: {termos}\n")
else:
    print(
        "Execute primeiro o treinamento do baseline "
        "(RUN_BASELINE_TRAINING=True)."
    )



# 10. Classificação — comparativo TF-IDF + SVM

Segundo classificador clássico no **mesmo corpus e split** do baseline. Implementação: `src/models/svm_tfidf.py` · treino: `make train-svm`.

Run de referência: `classification_svm_20260624-004348` (F1 macro teste ≈ **0,65**).


In [ ]:
# ============================================================
# 10. TF-IDF + SVM — comparativo (mesmo dataset do baseline)
# ============================================================

from src.models.svm_tfidf import build_svm

svm_params = {
    "ngram_max": 2,
    "min_df": 2,
    "max_features": 20000,
    "C": 1.0,
    "class_weight": "balanced",
    "kernel": "linear",
    "probability": True,
    "seed": SEED,
}

pipeline_svm = build_svm(**svm_params)

if RUN_SVM_TRAINING:
    print("
Treinando TF-IDF + SVM...")
    pipeline_svm.fit(dataset.train.texts, dataset.train.labels)
else:
    print("RUN_SVM_TRAINING=False: pulando treinamento SVM.")


def avaliar_svm(nome: str, split) -> dict:
    y_pred = pipeline_svm.predict(split.texts)
    m = compute_metrics(split.labels, list(y_pred), AREAS)
    return {"conjunto": nome, "modelo": "SVM", "f1_macro": m["f1_macro"], "accuracy": m["accuracy"]}


if RUN_SVM_TRAINING:
    desempenho_svm = pd.DataFrame(
        [
            avaliar_svm("treino", dataset.train),
            avaliar_svm("validação", dataset.val),
            avaliar_svm("teste", dataset.test),
        ]
    )
    display(desempenho_svm)

    comparativo = desempenho.assign(modelo="LogReg")[
        ["conjunto", "modelo", "f1_macro", "accuracy"]
    ]
    comparativo = pd.concat(
        [comparativo, desempenho_svm[["conjunto", "modelo", "f1_macro", "accuracy"]]],
        ignore_index=True,
    )
    print("
Comparativo LogReg vs SVM (F1 macro):")
    display(comparativo.pivot(index="conjunto", columns="modelo", values="f1_macro"))

    y_svm_test = pipeline_svm.predict(dataset.test.texts)
    print("
" + format_metrics(compute_metrics(dataset.test.labels, list(y_svm_test), AREAS)))


# Probabilidade de classe no baseline (LogReg)

A **Regressão Logística** retorna probabilidades via softmax sobre os logits de cada classe. A classe prevista é a de maior probabilidade (`argmax`), analogamente ao BERTimbau na Fase 2.

Isso permite interpretar a confiança da predição sem calibração adicional (diferente do SVM com `probability=True`, que usa Platt scaling).



In [ ]:
def prever_categoria_baseline(texto: str) -> tuple[str, float, dict[str, float]]:
    """Classifica um texto com o baseline TF-IDF + LogReg."""
    probs = pipeline_baseline.predict_proba([texto])[0]
    pred_id = int(np.argmax(probs))
    confianca = float(probs[pred_id])
    categoria = AREAS[pred_id]
    probabilidades = {AREAS[i]: float(probs[i]) for i in range(len(AREAS))}
    return categoria, confianca, probabilidades



In [ ]:
# Exemplo: predict_proba do baseline na tarefa de macroárea
texto_teste = dataset.test.texts[0]

probs = pipeline_baseline.predict_proba([texto_teste])[0]
pred_id = int(np.argmax(probs))

print("Probabilidades por classe (LogReg):")
for i, prob in enumerate(probs):
    print(f"  {AREAS[i]}: {prob:.4f}")
print("Classe prevista:", AREAS[pred_id])
print("\nTrecho de entrada (objeto_html):")
print(texto_teste[:500])

categoria, confianca, probabilidades = prever_categoria_baseline(texto_teste)
print(f"\nResumo: {categoria} (confiança {confianca:.2%})")



In [ ]:
texto_teste = dataset.test.texts[0]

categoria, confianca, probabilidades = prever_categoria_baseline(texto_teste)

print('Macroárea prevista:', categoria)
print('Confiança:', round(confianca, 4))
print('Probabilidades:')
print(probabilidades)


# Diagnóstico de overfitting na classificação por macroárea

Compare F1 macro entre treino, validação e teste. Gap grande treino → teste indica overfitting. O baseline oficial usa `objeto_html` e split 70/15/15 — referência: run `classification_baseline_20260608-190839` (F1 macro teste ≈ 0,74).



In [ ]:
# ============================================================
# 7.2 Diagnóstico treino vs validação vs teste
# ============================================================

if RUN_BASELINE_TRAINING and "desempenho" in globals():
    print("F1 macro por conjunto:")
    for _, row in desempenho.iterrows():
        print(f"  {row['conjunto']}: {row['f1_macro']:.4f}")

    gap = desempenho.loc[desempenho["conjunto"] == "treino", "f1_macro"].iloc[0]
    gap -= desempenho.loc[desempenho["conjunto"] == "teste", "f1_macro"].iloc[0]
    print(f"\nGap treino-teste (F1 macro): {gap:.4f}")

    if gap > 0.15:
        print("Atenção: gap elevado — possível overfitting.")
    else:
        print("Gap moderado — comportamento esperado para TF-IDF + LogReg.")
else:
    print("Execute a célula de treinamento do baseline primeiro.")



# 9. Definição do resumo em linguagem cidadã

O resumo em linguagem cidadã deve ser curto, direto e fiel ao edital. A proposta é que cada resumo tenha **um parágrafo**, com aproximadamente **80 a 130 palavras**, evitando jargões como “certame”, “objeto licitatório”, “instrumento convocatório” e “adjudicação”, salvo quando indispensáveis.

## Critérios do resumo ideal

O resumo deve conter:

1. **Objeto:** o que será comprado ou contratado.
2. **Órgão:** quem está conduzindo a contratação.
3. **Participação:** quem, em tese, pode participar.
4. **Valor ou escala:** valor homologado, quantidade ou dimensão, quando disponível.
5. **Alerta:** que o edital oficial deve ser consultado para regras completas.

## Exemplo de saída desejada

> Este edital trata da compra de medicamentos para atender necessidades da Secretaria de Estado de Saúde do Distrito Federal. Empresas fornecedoras do setor farmacêutico que cumpram os requisitos do edital podem participar da disputa. O valor informado na base é de R$ 62.800,00. O interessado deve consultar o edital completo para verificar prazos, documentos exigidos, condições de entrega e demais regras da contratação.


In [ ]:
# ============================================================
# 7. Baseline simples: resumo por regras
# ============================================================

def resumo_cidadao_baseline(row):
    orgao = row.get('orgao', 'órgão público')
    modalidade = row.get('modalidade', 'licitação')
    objeto = row.get('objeto_limpo', '')
    tipo = row.get('tipo', '')
    valor = row.get('total_homologado', None)

    if pd.isna(valor) or str(valor).strip() == '':
        valor_texto = 'O valor não foi identificado na planilha.'
    else:
        valor_texto = f'O valor informado na base é de R$ {valor}.'

    resumo = (
        f'Este edital é uma {modalidade.lower()} conduzida por {orgao}. '
        f'O objetivo é {objeto.lower()}. '
        f'Fornecedores que atuem com {str(tipo).lower()} e atendam às exigências do edital podem avaliar a participação. '
        f'{valor_texto} '
        f'O interessado deve consultar o edital completo para confirmar prazos, documentos exigidos, condições de entrega e demais regras da contratação.'
    )

    resumo = re.sub(r'\s+', ' ', resumo).strip()
    return resumo


licitacoes['resumo_baseline'] = licitacoes.apply(resumo_cidadao_baseline, axis=1)

display(licitacoes[['no_da_licitacao', 'orgao', 'objeto_limpo', 'resumo_baseline']].head(3))


# 10. Estratégia com LLM via engenharia de prompt

Uma estratégia prática para o projeto é usar um LLM com prompt estruturado. O prompt deve impor fidelidade ao edital, linguagem simples, um único parágrafo, proibição de inventar prazos, valores ou exigências não encontrados e aviso de que o resumo não substitui o edital oficial.

Essa abordagem é adequada quando o objetivo é gerar resumos de alta qualidade sem treinar um modelo do zero. O notebook abaixo deixa uma função genérica, sem incluir chave de API. O grupo pode adaptá-la ao provedor escolhido.


In [ ]:
# ============================================================
# 8. Prompt estruturado para LLM
# ============================================================

#PROMPT_LINGUAGEM_CIDADA = '''
#Você é um assistente especializado em transparência pública e linguagem cidadã.

#Tarefa: leia as informações de um edital de contratação pública e gere um resumo executivo em 1 único parágrafo, com linguagem simples, clara e acessível.

#O resumo deve responder, quando a informação estiver disponível:
#1. O que está sendo comprado ou contratado?
#2. Qual órgão conduz a contratação?
#3. Quem pode se interessar em participar?
#4. Há valor, prazo, entrega ou condição importante?
#5. Qual alerta o cidadão ou fornecedor deve observar?

#Regras obrigatórias:
#- Não invente informações ausentes.
#- Não use jargão jurídico desnecessário.
#- Não ultrapasse 130 palavras.
#- Informe que o edital completo deve ser consultado para regras e prazos oficiais.
#- Escreva em português do Brasil.

#Texto do edital ou registro:
#{texto}

#Resumo em linguagem cidadã:
#'''.strip()


#def montar_prompt_linguagem_cidada(texto):
#    texto = limitar_texto(texto, max_chars=12000)
#    return PROMPT_LINGUAGEM_CIDADA.format(texto=texto)


#exemplo_prompt = montar_prompt_linguagem_cidada(licitacoes['texto_minimo_para_resumo'].iloc[0])
#print(exemplo_prompt[:2500])


In [ ]:
# ============================================================
# 9. Função-modelo para uso de API de LLM
# ============================================================
# Esta célula é um esqueleto. Não coloque chaves de API diretamente no notebook.
# Use variáveis de ambiente ou secrets do Colab/GitHub.

# import os
# from openai import OpenAI
#
# client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
#
# def gerar_resumo_llm(texto, modelo='gpt-4o-mini'):
#     prompt = montar_prompt_linguagem_cidada(texto)
#     resposta = client.chat.completions.create(
#         model=modelo,
#         messages=[
#             {'role': 'system', 'content': 'Você resume documentos públicos em linguagem cidadã, com fidelidade e clareza.'},
#             {'role': 'user', 'content': prompt}
#         ],
#         temperature=0.2,
#         max_tokens=220
#     )
#     return resposta.choices[0].message.content.strip()
#
# # Teste em pequena amostra:
# # licitacoes_amostra = licitacoes.head(5).copy()
# # licitacoes_amostra['resumo_llm'] = licitacoes_amostra['texto_minimo_para_resumo'].apply(gerar_resumo_llm)
# # display(licitacoes_amostra[['n_da_licitacao', 'resumo_llm']])


# 12. Sumarização abstrativa com PTT5

Nesta versão, a sumarização local usa **PTT5** fine-tuned para sumarização em português. A tarefa continua sendo gerar um resumo de edital em linguagem cidadã, com foco em clareza, transparência e utilidade para cidadãos e pequenas empresas.

Modelo usado na geração local:

```text
recogna-nlp/ptt5-base-summ
```

Observação metodológica: a sumarização segue sendo uma trilha complementar ao classificador por macroárea. O ideal, em versão final do projeto, é comparar baseline por regras, PTT5 e LLM com prompt estruturado.


In [ ]:
# ============================================================
# 12. Sumarização abstrativa local com PTT5
# ============================================================
# Para executar, altere RUN_PTT5_SUMMARIZATION para True.
# PTT5 já ajustado para sumarização em português.
# ============================================================

if RUN_PTT5_SUMMARIZATION:

    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # Modelo PTT5 ajustado especificamente para sumarização em português
    PTT5_SUM_MODEL = "recogna-nlp/ptt5-base-summ"

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer_ptt5 = AutoTokenizer.from_pretrained(PTT5_SUM_MODEL)
    model_ptt5 = AutoModelForSeq2SeqLM.from_pretrained(PTT5_SUM_MODEL).to(device)

    print("Modelo PTT5 carregado:", PTT5_SUM_MODEL)
    print("Dispositivo:", device)

    def limitar_texto_seguro(texto, max_chars=6000):
        """
        Garante que o texto enviado ao modelo não fique excessivamente longo.
        Usa a função limitar_texto, se ela já existir no notebook.
        Caso contrário, aplica um corte simples por caracteres.
        """
        texto = "" if texto is None else str(texto)

        if "limitar_texto" in globals():
            return limitar_texto(texto, max_chars=max_chars)

        return texto[:max_chars]


    def gerar_resumo_ptt5(texto, max_input_tokens=512):
        """
        Gera resumo abstrativo em português usando PTT5 fine-tuned para sumarização.
        Esta função recebe apenas o conteúdo informativo do edital,
        sem prompt longo de instrução, para evitar que o modelo copie o comando.
        """

        if texto is None or len(str(texto).strip()) == 0:
            return "Texto insuficiente para gerar resumo."

        texto = limitar_texto_seguro(texto, max_chars=6000).strip()

        inputs = tokenizer_ptt5(
            texto,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_tokens
        ).to(device)

        with torch.no_grad():
            output_ids = model_ptt5.generate(
                **inputs,
                max_new_tokens=90,
                min_new_tokens=25,
                num_beams=4,
                no_repeat_ngram_size=3,
                repetition_penalty=1.15,
                length_penalty=1.0,
                early_stopping=True
            )

        resumo = tokenizer_ptt5.decode(
            output_ids[0],
            skip_special_tokens=True
        )

        return resumo.strip()


    def obter_valor_linha(linha, possiveis_colunas, padrao=""):
        """
        Busca o valor de uma linha considerando possíveis nomes de colunas.
        Isso deixa o código robusto caso a base use 'Órgão', 'Orgao',
        'orgao', 'Objeto', 'objeto' etc.
        """
        for coluna in possiveis_colunas:
            if coluna in linha.index:
                valor = linha.get(coluna, padrao)

                if pd.notna(valor):
                    return str(valor).strip()

        return padrao


    def gerar_texto_base_edital(linha):
        """
        Monta um texto informativo limpo a partir dos campos principais da licitação.
        Esse texto será enviado ao PTT5 para sumarização.
        """

        orgao = obter_valor_linha(
            linha,
            ["Órgão", "Orgao", "orgao", "orgao_nome", "nome_orgao", "unidade", "Unidade"],
            "órgão não informado"
        )

        modalidade = obter_valor_linha(
            linha,
            ["Modalidade", "modalidade"],
            "modalidade não informada"
        )

        situacao = obter_valor_linha(
            linha,
            ["Situação", "Situacao", "situacao"],
            "situação não informada"
        )

        tipo = obter_valor_linha(
            linha,
            ["Tipo", "tipo"],
            "tipo não informado"
        )

        objeto = obter_valor_linha(
            linha,
            ["Objeto", "objeto", "descricao", "Descrição", "Descricao"],
            "objeto não informado"
        )

        texto_base = f"""
        Órgão responsável: {orgao}.
        Modalidade da contratação: {modalidade}.
        Situação da licitação: {situacao}.
        Tipo de contratação: {tipo}.
        Objeto do edital: {objeto}.
        """

        return texto_base.strip()


    def gerar_resumo_cidadao_com_ptt5(linha):
        """
        Usa o PTT5 para resumir o conteúdo principal do edital
        e depois organiza a saída em linguagem cidadã.
        """

        orgao = obter_valor_linha(
            linha,
            ["Órgão", "Orgao", "orgao", "orgao_nome", "nome_orgao", "unidade", "Unidade"],
            "órgão não informado"
        )

        modalidade = obter_valor_linha(
            linha,
            ["Modalidade", "modalidade"],
            "modalidade não informada"
        )

        objeto = obter_valor_linha(
            linha,
            ["Objeto", "objeto", "descricao", "Descrição", "Descricao"],
            "objeto não informado"
        )

        texto_base = gerar_texto_base_edital(linha)

        resumo_modelo = gerar_resumo_ptt5(texto_base)

        resumo_cidadao = (
            f"O edital é conduzido por {orgao}, na modalidade {modalidade}. "
            f"Em linguagem simples, a contratação trata de {resumo_modelo.lower()} "
            f"Antes de participar, o interessado deve consultar o edital completo "
            f"para verificar prazos, documentos exigidos, condições de participação, "
            f"critérios de julgamento e regras de entrega."
        )

        return resumo_cidadao


    # ============================================================
    # Teste com a primeira licitação da base
    # ============================================================

    linha_teste = licitacoes.iloc[0]

    texto_base_teste = gerar_texto_base_edital(linha_teste)
    resumo_ptt5_teste = gerar_resumo_ptt5(texto_base_teste)
    resumo_cidadao_teste = gerar_resumo_cidadao_com_ptt5(linha_teste)

    print("\nTexto enviado ao PTT5:")
    print(texto_base_teste)

    print("\nResumo bruto gerado pelo PTT5:")
    print(resumo_ptt5_teste)

    print("\nResumo final em linguagem cidadã:")
    print(resumo_cidadao_teste)

else:
    print(
        "Sumarização com PTT5 não executada. "
        "Para gerar resumos com PTT5, altere RUN_PTT5_SUMMARIZATION para True."
    )

# 12. Criação de base de avaliação

Para avaliar a qualidade dos resumos, recomenda-se criar uma amostra manualmente anotada. Por exemplo:

- selecionar 50 a 100 editais;
- produzir manualmente um resumo de referência para cada um;
- comparar os resumos gerados com as referências.

## Métricas automáticas sugeridas

- **ROUGE-1:** sobreposição de unigramas com resumo de referência.
- **ROUGE-2:** sobreposição de bigramas.
- **ROUGE-L:** similaridade baseada em maior subsequência comum.
- **BERTScore ou similaridade semântica:** se o grupo optar por embeddings.

## Avaliação qualitativa sugerida

Criar uma rubrica de 1 a 5 para clareza, fidelidade, completude, ausência de alucinação e utilidade para cidadão ou pequeno fornecedor.


In [ ]:
# ============================================================
# 11. Estrutura para avaliação com ROUGE
# ============================================================
# É necessário ter uma coluna de resumo de referência, criada manualmente.

# from rouge_score import rouge_scorer
#
# def calcular_rouge(referencia, gerado):
#     scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
#     scores = scorer.score(str(referencia), str(gerado))
#     return {
#         'rouge1_f': scores['rouge1'].fmeasure,
#         'rouge2_f': scores['rouge2'].fmeasure,
#         'rougeL_f': scores['rougeL'].fmeasure,
#     }
#
# # aval = licitacoes_avaliacao.dropna(subset=['resumo_referencia', 'resumo_llm']).copy()
# # metricas = aval.apply(lambda r: calcular_rouge(r['resumo_referencia'], r['resumo_llm']), axis=1)
# # metricas_df = pd.DataFrame(list(metricas))
# # display(metricas_df.describe())


# 13. Análise de legibilidade e linguagem cidadã

Além de comparar com resumos de referência, o grupo pode medir características objetivas do texto gerado:

- número de palavras;
- número de frases;
- tamanho médio das frases;
- presença de termos técnicos;
- presença de alerta para consultar o edital completo;
- presença de objeto, órgão, valor e público interessado.

Essas medidas ajudam a demonstrar que o resumo ficou mais acessível do que o edital original.


In [ ]:
# ============================================================
# 12. Métricas simples de legibilidade
# ============================================================

TERMOS_TECNICOS = [
    'certame', 'adjudicação', 'homologação', 'instrumento convocatório',
    'contratante', 'contratada', 'habilitação', 'impugnação', 'sanções administrativas'
]


def metricas_legibilidade(texto):
    texto = str(texto)
    palavras = re.findall(r'\w+', texto.lower())
    frases = re.split(r'[.!?]+', texto)
    frases = [f.strip() for f in frases if f.strip()]

    n_palavras = len(palavras)
    n_frases = len(frases)
    media_palavras_frase = n_palavras / max(n_frases, 1)
    termos_encontrados = [t for t in TERMOS_TECNICOS if t in texto.lower()]

    return {
        'n_palavras': n_palavras,
        'n_frases': n_frases,
        'media_palavras_por_frase': media_palavras_frase,
        'n_termos_tecnicos': len(termos_encontrados),
        'termos_tecnicos': ', '.join(termos_encontrados)
    }


metricas_baseline = licitacoes['resumo_baseline'].head(10).apply(metricas_legibilidade)
metricas_baseline_df = pd.DataFrame(list(metricas_baseline))
display(metricas_baseline_df.describe())


# 14. Produto final esperado

O produto final do projeto pode ser uma tabela com os seguintes campos:

- `n_da_licitacao`;
- `orgao`;
- `modalidade`;
- `situacao`;
- `tipo`;
- `objeto_limpo`;
- `edital`;
- `texto_extraido`;
- `resumo_baseline`;
- `resumo_transformer` ou `resumo_llm`;
- métricas de legibilidade;
- avaliação manual.

A saída pode ser salva como CSV ou Parquet e disponibilizada no repositório do GitHub.


In [ ]:
# ============================================================
# 13. Exportação de resumos baseline (extrativo) + área (label proxy)
# ============================================================

colunas_saida = [
    "no_da_licitacao",
    "modalidade",
    "situacao",
    "orgao",
    "macroarea_gasto",
    "codigo_comprasnet",
    "tipo",
    "objeto_limpo",
    "edital",
    "total_homologado",
    "total_homologado_num",
    "resumo_baseline",
]

# Área proxy a partir do órgão (mesma lógica do corpus JSONL)
from src.preprocess.labels import area_for_orgao

saida = licitacoes.copy()
saida["macroarea_gasto"] = saida["orgao"].apply(area_for_orgao)

saida = saida[colunas_saida].copy()
saida_path = PROCESSED_DIR / "resumos_editais_linguagem_cidada_baseline.csv"
saida.to_csv(saida_path, index=False, encoding="utf-8-sig")

print("Arquivo exportado:", saida_path)
display(saida.head())



# 15. Metodologia proposta para o relatório final

## 15.1 Coleta

A coleta parte da planilha de licitações públicas de 2025. O script `run_collect.py` baixa os HTMLs; `run_preprocess.py` gera o corpus JSONL.

## 15.2 Pré-processamento

Entrada honesta para classificação: **`objeto_html`**. Auditoria de vazamento documentada em `docs/vazamento_de_label.md`.

## 15.3 Modelagem

1. **Baseline (Fase 1):** TF-IDF + Regressão Logística — métricas oficiais do grupo.
2. **BERTimbau (Fase 2):** fine-tuning — comparado ao baseline no relatório.
3. **Sumarização extrativa:** regras/regex (`src/summarize/extractive.py`).
4. **Sumarização abstrativa (Fase 3):** PTT5 neste notebook.

## 15.4 Avaliação

Classificação: F1 macro, matriz de confusão. Sumarização: ROUGE + rubrica humana 1–5.

## 15.5 Discussão crítica

Label proxy, corpus pequeno, limites de PDF, risco de alucinação em modelos abstrativos.



# 16. Riscos, vieses e uso responsável

O sistema deve ser apresentado como ferramenta auxiliar, não como substituto do edital oficial. O resumo pode facilitar a triagem, mas a decisão de participar de uma licitação depende da leitura integral do documento.

## Principais riscos

- **Alucinação:** o modelo pode incluir informação que não está no edital.
- **Omissão:** o resumo pode deixar de mencionar uma exigência importante.
- **Simplificação excessiva:** linguagem simples não pode alterar o sentido jurídico do documento.
- **Viés de acesso:** documentos escaneados ou mal formatados podem ser menos bem processados.
- **Dependência tecnológica:** uso de API externa pode gerar custo, limite de chamadas e questões de privacidade.

## Mitigações

- incluir regra para não inventar informação;
- manter link para o edital original;
- registrar confiança/qualidade do resumo;
- aplicar avaliação humana em amostra;
- comparar baseline, Transformer e LLM;
- documentar limitações de coleta e extração.


# 17. Estrutura sugerida do repositório

```text
sumarizacao-editais-linguagem-cidada/
│
├── README.md
├── requirements.txt
├── notebooks/
│   └── projeto_final_pln_sumarizacao_editais.ipynb
├── data/
│   ├── licitacoes2025.csv
│   ├── raw_editais/
│   └── processed/
│       └── resumos_editais_linguagem_cidada_baseline.csv
├── src/
│   ├── coleta_editais.py
│   ├── preprocessamento.py
│   ├── sumarizacao.py
│   └── avaliacao.py
└── slides/
    └── apresentacao_final.pdf
```

## Nome sugerido para o repositório

`sumarizacao-editais-linguagem-cidada`

## Descrição curta

> Projeto de PLN para resumir editais de contratação pública em linguagem cidadã, usando scraping, Transformers, LLMs e avaliação de clareza, fidelidade e utilidade pública.


# 18. Roteiro sugerido para apresentação de 10 minutos

1. **Problema real:** editais são públicos, mas pouco compreensíveis.
2. **Objetivo:** gerar resumos em linguagem cidadã.
3. **Dados:** planilha de licitações e links dos editais.
4. **Coleta:** scraping/download e extração de texto.
5. **Métodos:** baseline, Transformer/Seq2Seq e LLM com prompt.
6. **Avaliação:** ROUGE, legibilidade e rubrica humana.
7. **Resultados esperados:** resumos mais curtos, claros e úteis.
8. **Limitações:** alucinação, PDFs ruins, ausência de prazo em alguns registros.
9. **Impacto:** transparência, controle social e apoio a pequenos fornecedores.
10. **Próximos passos:** interface web, API pública e integração com portais de compras.


# 19. Referências iniciais para fundamentação

1. **TF-IDF + LogReg** — baseline clássico de categorização textual (Joachims 1998; sklearn Pipeline).
2. **BERTimbau** — Souza et al., modelo pré-treinado em português (Fase 2).
3. **PTT5** — modelo seq2seq para sumarização em português (Fase 3).
4. **Vazamento de label** — documentado em `docs/vazamento_de_label.md`.
5. **Lei 14.133** — marco de licitações e contratos administrativos.



# 20. Conclusão da proposta

A proposta de **Sumarização de Editais para Linguagem Cidadã** é adequada ao projeto final porque combina um problema público real, dados textuais coletáveis, aplicação concreta de PLN e discussão crítica sobre impacto social.

O projeto não se limita à execução técnica de um modelo. Ele busca demonstrar como técnicas de Deep Learning e PLN podem melhorar a transparência, reduzir barreiras de compreensão e apoiar a participação de cidadãos e pequenos fornecedores nas contratações públicas.

O resultado esperado é um pipeline reprodutível que transforme editais extensos e técnicos em resumos curtos, claros, fiéis e úteis, sempre preservando o link para o edital oficial e reconhecendo os limites do uso de IA generativa em contexto público.
